# Phase 10: Forecasting

Generate 12-week recursive forecasts per product using the trained Phase 8 Random Forest model.

## 1. Setup

In [1]:
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve project root whether executed from repo root or notebooks/ via nbconvert
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from models import get_feature_names

DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_OUTPUTS = PROJECT_ROOT / 'data' / 'outputs'
REPORTS_CHARTS = PROJECT_ROOT / 'reports' / 'charts'
MODELS_DIR = PROJECT_ROOT / 'models'

DATA_OUTPUTS.mkdir(parents=True, exist_ok=True)
REPORTS_CHARTS.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('Setup complete')

Setup complete


## 2. Load Data and Model

In [2]:
df = pd.read_csv(DATA_PROCESSED / 'feature_ready_dataset.csv')
df['Week_End_Date'] = pd.to_datetime(df['Week_End_Date'])
df = df.sort_values(['Product_ID', 'Week_End_Date']).reset_index(drop=True)

model_path = MODELS_DIR / 'best_model_rf.pkl'
with open(model_path, 'rb') as f:
    model = pickle.load(f)

feature_cols = get_feature_names(df)

print(f'Data shape: {df.shape}')
print(f'Products: {df["Product_ID"].nunique()}')
print(f'Date range: {df["Week_End_Date"].min().date()} to {df["Week_End_Date"].max().date()}')
print(f'Model: {type(model).__name__}')
print(f'Feature count: {len(feature_cols)}')
print(f'Features: {feature_cols}')

Data shape: (2600, 30)
Products: 25
Date range: 2023-01-07 to 2024-12-28
Model: RandomForestRegressor
Feature count: 23
Features: ['Year', 'Month', 'Quarter', 'ISO_Week', 'Week_Number', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Lag_8', 'Lag_12', 'Rolling_Mean_4', 'Rolling_Mean_8', 'Rolling_Std_4', 'Rolling_Max_4', 'Rolling_Min_4', 'Discount_Flag', 'Returns_Rate', 'Inventory_Ratio', 'Promo_Flag', 'Holiday_Flag', 'Rainfall', 'Avg_Temperature']


## 3. Recursive Weekly Forecast

In [3]:
def build_forecast_row(product_df, history, forecast_date):
    """Build one future row using only last known values and recursive forecast history."""
    last_row = product_df.iloc[-1].copy()
    row = last_row.copy()
    
    row['Week_End_Date'] = forecast_date
    row['Year'] = forecast_date.year
    row['Month'] = forecast_date.month
    row['Quarter'] = forecast_date.quarter
    row['ISO_Week'] = int(forecast_date.isocalendar().week)
    row['Week_Number'] = int(forecast_date.strftime('%U'))
    row['Month_Start'] = forecast_date.to_period('M').start_time
    row['Month_End'] = forecast_date.to_period('M').end_time
    
    for lag in [1, 2, 3, 4, 8, 12]:
        row[f'Lag_{lag}'] = history[-lag] if len(history) >= lag else np.nan
    
    previous_4 = pd.Series(history[-4:], dtype='float64')
    previous_8 = pd.Series(history[-8:], dtype='float64')
    row['Rolling_Mean_4'] = previous_4.mean() if len(previous_4) else np.nan
    row['Rolling_Mean_8'] = previous_8.mean() if len(previous_8) else np.nan
    row['Rolling_Std_4'] = previous_4.std() if len(previous_4) > 1 else np.nan
    row['Rolling_Max_4'] = previous_4.max() if len(previous_4) else np.nan
    row['Rolling_Min_4'] = previous_4.min() if len(previous_4) else np.nan
    
    return row


forecast_rows = []
horizon = 12

for product_id, product_df in df.groupby('Product_ID', sort=True):
    product_df = product_df.sort_values('Week_End_Date').reset_index(drop=True)
    history = product_df['Qty_Sold'].astype(float).tolist()
    last_date = product_df['Week_End_Date'].max()
    
    for step in range(1, horizon + 1):
        forecast_date = last_date + pd.Timedelta(weeks=step)
        row = build_forecast_row(product_df, history, forecast_date)
        
        X_future = row[feature_cols].to_frame().T.astype(float)
        X_future = X_future.fillna(0.0)
        forecast_qty = float(model.predict(X_future)[0])
        forecast_qty = max(0.0, forecast_qty)
        
        history.append(forecast_qty)
        row['Qty_Sold'] = np.nan
        row['Forecast_Qty_Sold'] = forecast_qty
        row['Forecast_Step'] = step
        forecast_rows.append(row)

forecast_weekly = pd.DataFrame(forecast_rows)

output_cols = [
    'Product_ID', 'Week_End_Date', 'Forecast_Step', 'Forecast_Qty_Sold',
    'Demand_Class', 'Year', 'Month', 'Quarter', 'ISO_Week', 'Week_Number'
]
forecast_weekly = forecast_weekly[output_cols].sort_values(['Product_ID', 'Week_End_Date']).reset_index(drop=True)

expected_rows = df['Product_ID'].nunique() * horizon
if len(forecast_weekly) != expected_rows:
    raise ValueError(f'Expected {expected_rows} forecast rows, found {len(forecast_weekly)}')

print(f'Weekly forecast shape: {forecast_weekly.shape}')
forecast_weekly.head(15)

C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X 

C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X 

C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X 

C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X 

C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X 

...

C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X 

C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X 

Weekly forecast shape: (300, 10)


C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(
C:\Users\welcome\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


,Product_ID,Week_End_Date,Forecast_Step,Forecast_Qty_Sold,Demand_Class,Year,Month,Quarter,ISO_Week,Week_Number
0,TE001,2025-01-04,1,76.376577,Smooth,2025,1,1,1,0
1,TE001,2025-01-11,2,76.376577,Smooth,2025,1,1,2,1
2,TE001,2025-01-18,3,76.456938,Smooth,2025,1,1,3,2
3,TE001,2025-01-25,4,76.376577,Smooth,2025,1,1,4,3
4,TE001,2025-02-01,5,76.662651,Smooth,2025,2,1,5,4
5,TE001,2025-02-08,6,76.662651,Smooth,2025,2,1,6,5
6,TE001,2025-02-15,7,76.662651,Smooth,2025,2,1,7,6
7,TE001,2025-02-22,8,76.662651,Smooth,2025,2,1,8,7
8,TE001,2025-03-01,9,76.729979,Smooth,2025,3,1,9,8
9,TE001,2025-03-08,10,76.729979,Smooth,2025,3,1,10,9


## 4. Monthly Forecast

In [4]:
forecast_monthly = forecast_weekly.copy()
forecast_monthly['Forecast_Month'] = forecast_monthly['Week_End_Date'].dt.to_period('M').astype(str)

forecast_monthly = (
    forecast_monthly
    .groupby(['Product_ID', 'Forecast_Month'], as_index=False)
    .agg(
        Forecast_Qty_Sold=('Forecast_Qty_Sold', 'sum'),
        Forecast_Weeks=('Forecast_Qty_Sold', 'size'),
        Demand_Class=('Demand_Class', 'first')
    )
)

print(f'Monthly forecast shape: {forecast_monthly.shape}')
forecast_monthly.head(15)

Monthly forecast shape: (75, 5)


,Product_ID,Forecast_Month,Forecast_Qty_Sold,Forecast_Weeks,Demand_Class
0,TE001,2025-01,305.586669,4,Smooth
1,TE001,2025-02,306.650602,4,Smooth
2,TE001,2025-03,306.981011,4,Smooth
3,TE002,2025-01,198.890793,4,Smooth
4,TE002,2025-02,201.541718,4,Smooth
5,TE002,2025-03,203.949718,4,Smooth
6,TE003,2025-01,236.185310,4,Smooth
7,TE003,2025-02,235.835326,4,Smooth
8,TE003,2025-03,235.961657,4,Smooth
9,TE004,2025-01,432.428745,4,Smooth


## 5. Save Outputs

In [5]:
weekly_path = DATA_OUTPUTS / 'forecast_weekly.csv'
monthly_path = DATA_OUTPUTS / 'forecast_monthly.csv'

forecast_weekly.to_csv(weekly_path, index=False)
forecast_monthly.to_csv(monthly_path, index=False)

print(f'Saved: {weekly_path}')
print(f'Saved: {monthly_path}')

Saved: D:\Final Project\data\outputs\forecast_weekly.csv
Saved: D:\Final Project\data\outputs\forecast_monthly.csv


## 6. Forecast Charts

In [6]:
actual_total = (
    df.groupby('Week_End_Date', as_index=False)['Qty_Sold']
    .sum()
    .tail(12)
)
forecast_total = forecast_weekly.groupby('Week_End_Date', as_index=False)['Forecast_Qty_Sold'].sum()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(actual_total['Week_End_Date'], actual_total['Qty_Sold'], marker='o', label='Actual')
ax.plot(forecast_total['Week_End_Date'], forecast_total['Forecast_Qty_Sold'], marker='o', label='Forecast')
ax.axvline(df['Week_End_Date'].max(), color='gray', linestyle='--', linewidth=1)
ax.set_title('Forecast vs Actual Demand')
ax.set_xlabel('Week End Date')
ax.set_ylabel('Qty Sold')
ax.legend()
plt.tight_layout()

forecast_vs_actual_path = REPORTS_CHARTS / 'forecast_vs_actual.png'
fig.savefig(forecast_vs_actual_path, dpi=300, bbox_inches='tight')
plt.close(fig)

top_product = df.groupby('Product_ID')['Qty_Sold'].sum().idxmax()
top_actual = df[df['Product_ID'] == top_product].tail(12)
top_forecast = forecast_weekly[forecast_weekly['Product_ID'] == top_product]

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(top_actual['Week_End_Date'], top_actual['Qty_Sold'], marker='o', label='Actual')
ax.plot(top_forecast['Week_End_Date'], top_forecast['Forecast_Qty_Sold'], marker='o', label='Forecast')
ax.axvline(df['Week_End_Date'].max(), color='gray', linestyle='--', linewidth=1)
ax.set_title(f'Top Product Forecast: {top_product}')
ax.set_xlabel('Week End Date')
ax.set_ylabel('Qty Sold')
ax.legend()
plt.tight_layout()

top_product_path = REPORTS_CHARTS / 'top_product_forecast.png'
fig.savefig(top_product_path, dpi=300, bbox_inches='tight')
plt.close(fig)

print(f'Saved: {forecast_vs_actual_path}')
print(f'Saved: {top_product_path}')

Saved: D:\Final Project\reports\charts\forecast_vs_actual.png
Saved: D:\Final Project\reports\charts\top_product_forecast.png


## Phase 10 Complete